In [5]:
import os
import json
import numpy as np
import librosa
import random

# Configuration
AUDIO_FILE = "C:\\Users\\maste\\OneDrive\\Desktop\\ASfuera\\wetransfer_1-jpg_2022-11-25_1720\\Rococo_In_Bloom\\songs\\Automation\\Coppélia-Valse_de_la_poupee.mp3"
OUTPUT_FILE = "chart.json"
LANE_COUNT = 5
MIN_GAP_MS = 250  # Sets difficulty to "Hard" - approx. 4 notes/sec max
SR = 22050  # Sample Rate
WINDOW_MS = 80  # Analysis window size around each onset
SEED = 123  # For reproducibility

np.random.seed(SEED)
random.seed(SEED)

In [6]:
# Load Audio & Detect Onsets
y, sr = librosa.load(AUDIO_FILE, sr=SR, mono=True)

onset_times = librosa.onset.onset_detect(y=y, sr=sr, units="time")
sorted_onsets = np.sort(onset_times)

accepted_onsets = []
last_onset = -np.inf
min_gap_s = MIN_GAP_MS / 1000.0
for onset in sorted_onsets:
    if onset - last_onset >= min_gap_s:
        accepted_onsets.append(float(onset))
        last_onset = onset

print(f"Detected onsets: {len(onset_times)} | Accepted onsets: {len(accepted_onsets)}")

Detected onsets: 387 | Accepted onsets: 264


In [7]:
# Feature Extraction (The "Smart Map")
features = []
window_samples = int((WINDOW_MS / 1000.0) * SR)
half_window = max(1, window_samples // 2)

for onset in accepted_onsets:
    center = int(onset * sr)
    start = max(0, center - half_window)
    end = min(len(y), center + half_window)
    window = y[start:end]

    if window.size == 0:
        features.append(0.0)
        continue

    centroid = librosa.feature.spectral_centroid(y=window, sr=sr)
    value = np.nanmean(centroid) if centroid.size > 0 else np.nan

    if np.isnan(value) or np.isinf(value):
        spectrum = np.abs(np.fft.rfft(window))
        freqs = np.fft.rfftfreq(window.size, d=1.0 / sr)
        if spectrum.size > 0:
            peak_idx = int(np.argmax(spectrum))
            value = float(freqs[peak_idx])
        else:
            value = 0.0

    features.append(float(value))

print(f"Extracted features for {len(features)} onsets.")

Extracted features for 264 onsets.


c:\Users\maste\AppData\Local\Programs\Python\Python311\Lib\site-packages\librosa\core\spectrum.py:266: UserWarning: n_fft=2048 is too large for input signal of length=1764
  warnings.warn(


In [8]:
# Normalization & Lane Mapping
features_array = np.array(features, dtype=float)

if features_array.size == 0:
    normalized = np.array([], dtype=float)
    mapped_lanes = []
else:
    p5, p95 = np.percentile(features_array, [5, 95])
    if p5 == p95:
        clamped = np.full_like(features_array, p5)
    else:
        clamped = np.clip(features_array, p5, p95)

    denominator = p95 - p5 if p95 != p5 else 1.0
    normalized = np.clip((clamped - p5) / denominator, 0.0, 1.0)
    mapped_lanes = (normalized * LANE_COUNT).astype(int)
    mapped_lanes = np.clip(mapped_lanes, 0, LANE_COUNT - 1).tolist()

print(f"Lane mapping complete. Sample lanes: {mapped_lanes[:10] if mapped_lanes else 'None'}")

Lane mapping complete. Sample lanes: [2, 1, 0, 0, 0, 0, 0, 3, 4, 4]


In [9]:
# Flow Logic (Constraint Solver)
final_lanes = []
prev_lane = None
prev_move = 0  # -1 for left, +1 for right, 0 for none

for lane in mapped_lanes:
    chosen_lane = lane
    if prev_lane is not None and lane == prev_lane:
        preferred_direction = -prev_move if prev_move != 0 else 1
        direction_order = [preferred_direction, -preferred_direction]
        found = False
        for direction in direction_order:
            for offset in range(1, LANE_COUNT):
                candidate = prev_lane + offset * direction
                if 0 <= candidate < LANE_COUNT:
                    chosen_lane = candidate
                    found = True
                    break
            if found:
                break
    move_direction = 0 if prev_lane is None else chosen_lane - prev_lane
    prev_lane = chosen_lane
    prev_move = int(np.sign(move_direction))
    final_lanes.append(int(chosen_lane))

print(f"Applied flow constraints. Sample final lanes: {final_lanes[:10] if final_lanes else 'None'}")

Applied flow constraints. Sample final lanes: [2, 1, 0, 1, 0, 1, 0, 3, 4, 3]


In [10]:
# Export & Summary
notes = []
for onset, lane in zip(accepted_onsets, final_lanes):
    time_ms = int(round(onset * 1000))
    notes.append({
        "time": time_ms,
        "lane": int(lane),
        "type": "normal",
    })

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(notes, f, indent=2)

duration_seconds = len(y) / sr if len(y) > 0 else 0.0
notes_per_second = (len(notes) / duration_seconds) if duration_seconds > 0 else 0.0
lane_distribution = np.bincount(final_lanes, minlength=LANE_COUNT) if final_lanes else np.zeros(LANE_COUNT, dtype=int)

print(f"Chart exported to: {OUTPUT_FILE}")
print(f"Total Notes: {len(notes)}")
print(f"Notes Per Second: {notes_per_second:.2f}")
print("Lane Distribution:")
for lane_idx, count in enumerate(lane_distribution):
    print(f"  Lane {lane_idx}: {int(count)}")

Chart exported to: chart.json
Total Notes: 264
Notes Per Second: 2.13
Lane Distribution:
  Lane 0: 51
  Lane 1: 75
  Lane 2: 68
  Lane 3: 47
  Lane 4: 23
